In [1]:
from pathlib import Path
import json, ast, hashlib, sys, platform
from typing import Dict, Any
import numpy as np
import pandas as pd
import h5py
from scipy.signal import butter, sosfilt, lfilter
sys.dont_write_bytecode=True
ROOT=(next(p for p in [Path.cwd().resolve(), *Path.cwd().resolve().parents] if (p/'dataset_2.csv').is_file()) / 'cnn/cnn wave')
V2=ROOT/'eew_causal_v2'; V3=ROOT/'eew_causal_v3'; OUT=V3/'codex_final_verification'
def readj(p): return json.loads(p.read_text(encoding='utf-8'))
def sha(p):
 h=hashlib.sha256()
 with p.open('rb') as f:
  for b in iter(lambda:f.read(4194304),b''): h.update(b)
 return h.hexdigest()
# Restrict inventory to v3 artifacts and specifically referenced dependencies.
files=[p for folder in [V3,V3/'configs',V3/'dataset',V3/'gemini_validation'] for p in folder.iterdir() if p.is_file()]
SIM=ROOT/'data wave/synthetic_waveform_simulator/output'
files += [V2/'02_generate_causal_eew_dataset.ipynb',V2/'codex_independent_audit/dataset_audit.ipynb',V2/'codex_independent_audit/audit_results.json',SIM/'phase2_travel_time/metadata_with_theoretical_arrivals.csv',SIM/'phase11_preprocessed_dataset/split_indices.csv',SIM/'phase10_classification_dataset/earthquake_noise_mapping.csv']
before={str(p):sha(p) for p in files}
R={'date':'2026-11-13','checks':{},'environment':{'python':platform.python_version(),'numpy':np.__version__,'h5py':h5py.__version__},'input_hashes':before}
def check(name,ok,evidence): R['checks'][name]={'status':'PASS' if ok else 'FAIL','evidence':evidence}
params=readj(V3/'configs/simulation_v3.json')['parameters']
# Execute only definition/calibration region, excluding all original file-writing setup and configuration export.
src=''.join(readj(V3/'01_causal_simulator_v3.ipynb')['cells'][1]['source'])
v2source=''.join(readj(V2/'02_generate_causal_eew_dataset.ipynb')['cells'][2]['source'])
v2tree=ast.parse(v2source)
v2classes=ast.unparse(ast.Module(body=[x for x in v2tree.body if isinstance(x,ast.ClassDef)],type_ignores=[]))
exec(compile(src[src.index('fs=100.;DT=.01'):src.index('configuration={')],'v3_read_only_definitions','exec'))
meta=pd.read_csv(SIM/'phase2_travel_time/metadata_with_theoretical_arrivals.csv').set_index('source_id')
assign=pd.read_csv(SIM/'phase11_preprocessed_dataset/split_indices.csv').set_index('combined_index')
noise_map=pd.read_csv(SIM/'phase10_classification_dataset/earthquake_noise_mapping.csv').set_index('combined_index')
print('Read-only simulator loaded; input hashes recorded',len(before))


Read-only simulator loaded; input hashes recorded 26


In [2]:
# Independent regression controls: instrument the original v2 source ourselves.
def oldgen(mode):
 s=v2classes
 if mode=='noise': s=s.replace('filtered_noise = sosfilt(self.noise_sos, raw_noise)','raw_noise[self.time_axis >= tp_sec + 8.] *= 4.\n        filtered_noise = sosfilt(self.noise_sos, raw_noise)')
 if mode=='coda': s=s.replace('max_env_p = np.max(env_p)', 'env_p[tau_p >= 4.] *= 4.\n        max_env_p = np.max(env_p)')
 ns=dict(globals());exec(compile(s,'v2_regression','exec'),ns);return ns['CausalEEWGenerator'](params)
def diff(a,b): return float(np.max(np.abs(np.asarray(a,dtype=float)-np.asarray(b,dtype=float))))
controls=[]; tests=[]
for case in readj(V2/'codex_independent_audit/audit_results.json')['counterfactuals']:
 m=meta.loc[case['source_id']]
 kw=dict(magnitude=float(m.Magnitude_Value),hypocentral_distance_km=float(m.hypocentral_distance_km),theoretical_sp_sec=float(m.theoretical_sp_time_seconds),seed=int(case['seed']),target_noise_rms=.05)
 old=oldgen('base').synthesize_earthquake_window(**kw)
 controls.append({'source_id':case['source_id'],'seed':kw['seed'],**{mode:diff(old['w3_norm'],oldgen(mode).synthesize_earthquake_window(**kw)['w3_norm']) for mode in ['noise','coda']}})
 base=CausalEEWGeneratorV3(params).synthesize_earthquake_window(**kw);p=base['p_sample'];s=base['s_sample']
 def compare(name,other,stop,tail=True):
  a=base['parent_raw'];b=other['parent_raw'];n=min(len(a),len(b))
  an=(a-base['context_mean'])/base['context_std'];bn=(b-other['context_mean'])/other['context_std']
  tests.append({'source_id':case['source_id'],'seed':kw['seed'],'test':name,'stop_exclusive':stop,'raw':diff(a[:stop],b[:stop]),'normalized':diff(an[:stop],bn[:stop]),'changed_tail':bool(np.any(a[stop:n]!=b[stop:n])) if tail else None})
 g=CausalEEWGeneratorV3(params);g.sp_ratio*=4;compare('S amplitude x4',g.synthesize_earthquake_window(**kw),s)
 for name,key,after in [('future P coda','p_coda',4.),('future noise','noise',8.),('future S coda','s_coda',5.)]:
  g=CausalEEWGeneratorV3(params);g.interventions={key:(after,4.)}
  stop=int(np.ceil((base['ts_sec']+after)*100)) if key=='s_coda' else p+int(after*100)
  compare(name,g.synthesize_earthquake_window(**kw),stop)
 for duration in [12.,140.]:
  other=CausalEEWGeneratorV3({**params,'duration_seconds':duration}).synthesize_earthquake_window(**kw)
  compare('duration '+str(duration),other,min(len(base['parent_raw']),len(other['parent_raw'])),False)
 for t in [1,2,3]:
  altered=base['parent_raw'].copy();altered[p+t*100:]+=123.
  other=normalize_windows(altered,p)
  tests.append({'source_id':case['source_id'],'seed':kw['seed'],'test':f'normalization cutoff {t}s','raw':0.,'normalized':diff(base[f'w{t}_norm'],other[f'w{t}_norm'])})
R['v2_controls']=controls;R['earthquake_counterfactuals']=tests
maxima={k:max(x[k] for x in controls) for k in ['noise','coda']}
check('v2 regression controls',abs(maxima['noise']-361.790329)<1e-6 and abs(maxima['coda']-99.573059)<1e-6,maxima)
check('v3 earthquake causality',all(x['raw']==x['normalized']==0 and x.get('changed_tail') is not False for x in tests),{'cases':len(controls),'tests':len(tests),'max_difference':max(max(x['raw'],x['normalized']) for x in tests)})
nt=[]
for profile in ['gaussian_bandlimited','colored_pink','modulated_nonstationary','transient_disturbances']:
 for seed in [19,71,1234,9099]:
  base=CausalNoiseWindowGeneratorV3(params).generate_noise_window(profile,seed);ref=base['reference_sample']
  for duration in [12.,140.,100.]:
   g=CausalNoiseWindowGeneratorV3({**params,'duration_seconds':duration})
   if duration==100.:g.interventions={'noise':(3.,4.)}
   other=g.generate_noise_window(profile,seed);stop=ref+300 if duration==100. else min(len(base['parent_raw']),len(other['parent_raw']))
   nt.append({'profile':profile,'seed':seed,'duration':duration,'raw':diff(base['parent_raw'][:stop],other['parent_raw'][:stop]),'normalized':diff(base['w3_norm'],other['w3_norm'])})
R['noise_counterfactuals']=nt
check('v3 noise causality',all(x['raw']==x['normalized']==0 for x in nt),{'tests':len(nt)})
print('Counterfactual checks', {k:v['status'] for k,v in R['checks'].items()})


Counterfactual checks {'v2 regression controls': 'PASS', 'v3 earthquake causality': 'PASS', 'v3 noise causality': 'PASS'}


In [3]:
counts={};eligible={t:0 for t in [1,2,3]};identities={};parent_hashes={};replays=[];failures=[];subnormals=0
manifest=pd.read_csv(V3/'configs/split_manifest.csv')
def require(ok,msg):
 if not ok: failures.append(msg)
for split,n in [('train',11324),('validation',2426),('test',2428)]:
 with h5py.File(V3/'dataset'/f'eew_dataset_{split}.h5','r') as f:
  d={k:f[k][:] for k in f if f[k].ndim==1}
  for k,v in list(d.items()):
   if v.dtype.kind=='O':d[k]=np.array([x.decode() if isinstance(x,bytes) else str(x) for x in v])
  y=d['labels'];eq=y==1;counts[split]={'total':len(y),'earthquakes':int(eq.sum()),'noise':int((~eq).sum())}
  require(len(y)==n and set(y)=={0,1},split+' count/labels')
  require(f.attrs['sampling_rate_hz']==100 and f['parent_raw'].shape==(n,10000),split+' sampling/parents')
  require(np.array_equal(d['magnitude_mask'],eq) and np.all(d['magnitude'][~eq]==-1),split+' base magnitude mask')
  a=assign.loc[d['combined_index']]
  require(np.array_equal(a.label,y) and np.all(a.split==split) and np.array_equal(a.pair_id,d['pair_id']),split+' source split mapping')
  m=meta.loc[d['source_id'][eq]]
  require(np.array_equal(m.Magnitude_Value.to_numpy().astype('float32'),d['magnitude'][eq]),split+' catalog magnitude alignment')
  require(np.array_equal(m.theoretical_sp_time_seconds.to_numpy(),d['sp_seconds'][eq]),split+' physical SP')
  require(np.array_equal(d['p_arrival_sample'][eq],d['reference_sample'][eq]) and np.all(d['p_arrival_seconds'][eq]==d['p_arrival_sample'][eq]/100),split+' P annotations')
  require(np.array_equal(np.ceil(d['s_arrival_seconds'][eq]*100).astype(int),d['s_arrival_sample'][eq]),split+' S first grid annotations')
  require(np.all(d['s_arrival_seconds'][eq]==d['p_arrival_seconds'][eq]+d['sp_seconds'][eq]),split+' S physical time')
  require(all(np.all(d[k][~eq]==-1) for k in ['p_arrival_sample','s_arrival_sample','p_arrival_seconds','s_arrival_seconds']),split+' noise arrival sentinels')
  for k in f:
   if f[k].dtype.kind in 'biuf':
    for start in range(0,n,256):
     arr=f[k][start:start+256];require(np.isfinite(arr).all(),split+' nonfinite '+k)
     if arr.dtype.kind=='f':subnormals+=int(((np.abs(arr)>0)&(np.abs(arr)<np.finfo(arr.dtype).tiny)).sum())
  for t in [1,2,3]:
   pre=eq & (d['reference_sample']+t*100<d['s_arrival_sample'])
   eligible[t]+=int(pre.sum())
   require(np.array_equal(d[f'pre_s_valid_{t}s'],pre) and np.array_equal(d[f'magnitude_mask_{t}s'],pre) and np.array_equal(d[f'classification_mask_{t}s'],(~eq)|pre),split+f' masks {t}')
   require(f[f'windows_{t}s_raw'].shape==(n,100+t*100) and f[f'windows_{t}s_norm'].shape==(n,100+t*100),split+f' window shape {t}')
  hashes=[]
  for start in range(0,n,128):
   parents=f['parent_raw'][start:start+128]
   for j,parent in enumerate(parents,start):
    ref=int(d['reference_sample'][j]);pre=parent[ref-100:ref];mean=float(pre.mean());std=float(pre.std())+1e-8
    require(mean==d['context_mean'][j] and std==d['context_std'][j],split+f' context {j}')
    h=hashlib.sha256(parent.tobytes()).hexdigest();hashes.append(h);require(h==d['parent_sha256'][j],split+f' parent hash {j}')
   refs=d['reference_sample'][start:start+len(parents)]
   for t in [1,2,3]:
    raw=np.stack([p[int(ref)-100:int(ref)+t*100] for p,ref in zip(parents,refs)])
    norm=np.stack([(r-float(p[int(ref)-100:int(ref)].mean()))/(float(p[int(ref)-100:int(ref)].std())+1e-8) for r,p,ref in zip(raw,parents,refs)])
    require(np.array_equal(raw,f[f'windows_{t}s_raw'][start:start+len(parents)]),split+f' raw slices {start}/{t}')
    require(np.array_equal(norm,f[f'windows_{t}s_norm'][start:start+len(parents)]),split+f' norm slices {start}/{t}')
    require(np.array_equal(raw[:,100:],f[f'windows_pure_post_p_{t}s'][start:start+len(parents)]),split+f' post slices {start}/{t}')
  parent_hashes[split]=set(hashes)
  identities[split]={k:set(d[k]) for k in ['source_id','pair_id','combined_index','parent_id','background_id','background_sha256']}
  # Stratify independent saved-parent replay by magnitude, SP, and every noise profile.
  indices=set()
  for key in ['magnitude','sp_seconds']:
   ix=np.flatnonzero(eq);order=ix[np.argsort(d[key][ix])];indices.update(order[np.linspace(0,len(order)-1,7,dtype=int)])
  for profile in np.unique(d['noise_profile'][~eq]):indices.update(np.flatnonzero((~eq)&(d['noise_profile']==profile))[:2])
  for i in sorted(indices):
   if eq[i]:
    mm=meta.loc[d['source_id'][i]];out=CausalEEWGeneratorV3(params).synthesize_earthquake_window(float(mm.Magnitude_Value),float(mm.hypocentral_distance_km),float(mm.theoretical_sp_time_seconds),int(d['seed'][i]),.05)
   else:
    nm=noise_map.loc[d['combined_index'][i]]
    require(int(nm.seed)==d['seed'][i] and nm.noise_profile_type==d['noise_profile'][i],split+' noise mapping')
    out=CausalNoiseWindowGeneratorV3(params).generate_noise_window(str(nm.noise_profile_type),int(nm.seed),.05)
   err=diff(out['parent_raw'],f['parent_raw'][i]);replays.append({'split':split,'row':int(i),'source_id':d['source_id'][i],'parent_max_difference':err})
   require(err==0,split+f' replay {i}')
  # Manifest association checked for every row.
  sm=manifest[manifest.split==split].sort_values('row')
  for k in ['combined_index','source_id','seed','pair_id'] :require(np.array_equal(sm[k].to_numpy(),d[k]),split+' manifest '+k)
check('saved HDF5 arrays and source alignment',not failures,{'failures':failures,'counts':counts,'records':sum(x['total'] for x in counts.values()),'subnormal_values':subnormals})
check('eligible counts',eligible=={1:8089,2:8056,3:7996},eligible)
overlap={}
for a,b in [('train','validation'),('train','test'),('validation','test')]:
 overlap[a+'/'+b]={k:len(identities[a][k]&identities[b][k]) for k in identities[a]}
 overlap[a+'/'+b]['actual_parent_hash']=len(parent_hashes[a]&parent_hashes[b])
check('split disjointness',all(v==0 for row in overlap.values() for v in row.values()),overlap)
R['saved_parent_replays']=replays
check('saved simulator replay',all(x['parent_max_difference']==0 for x in replays),{'records':len(replays),'max_difference':max(x['parent_max_difference'] for x in replays)})
npzs=list((V3/'dataset').glob('*.npz'))
R['checks']['NPZ verification']={'status':'UNKNOWN','evidence':'No NPZ archive exists in the v3 dataset directory. No v2 archive substituted.'}
print('Saved data checks complete',counts,'eligible',eligible,'failures',len(failures),'replays',len(replays))


Saved data checks complete {'train': {'total': 11324, 'earthquakes': 5662, 'noise': 5662}, 'validation': {'total': 2426, 'earthquakes': 1213, 'noise': 1213}, 'test': {'total': 2428, 'earthquakes': 1214, 'noise': 1214}} eligible {1: 8089, 2: 8056, 3: 7996} failures 0 replays 66


In [4]:
check('model input provenance',True,'Inspected generation path and exact saved-parent replay: inputs are waveform-only (record,time) arrays. Labels, magnitudes, IDs, P/S annotations and masks are separate datasets. Simulator source parameters affect physical synthetic signals; no target channels are concatenated. This is source-path evidence, not a guarantee against arbitrary statistical shortcuts.')
check('oracle alignment and scientific scope',True,'P reference is theoretical/oracle and occurs at window index 100. Context normalization uses only the preceding 100 samples. Horizon windows are half-open; eligible examples have reference+100*horizon < ceil(S_seconds*100). S timing is privileged eligibility metadata, never an input feature. This is synthetic post-trigger analysis, not a demonstrated streaming trigger or operational EEW system.')
check('regression masking contract',True,'All saved magnitude masks equal label==1; per-horizon masks additionally require pre-S eligibility. Noise targets are -1 and masked out. Ineligible EQ rows remain stored and MUST be excluded using horizon masks. No training consumer exists in this audit, so runtime enforcement by a future trainer is unverified.')
after={p:sha(Path(p)) for p in before}
check('existing input preservation',before==after,{'files':len(before),'changed':[p for p in before if before[p]!=after[p]],'scope':'All read dependencies and v3 top-level/config/dataset/Gemini artifacts hashed before and after; no project-wide scan.'})
R['decision']='BLOCKED — critical failure' if any(x['status']=='FAIL' for x in R['checks'].values()) else 'CONDITIONAL — NPZ artifact verification remains outstanding'
R['limitations']=['No NPZ saved in v3 dataset directory; its equality, labels and masks cannot be verified.','HDF5 is suitable for controlled synthetic oracle-aligned CNN experiments if per-horizon masks are used; no operational EEW approval.','No training, dataset regeneration, or data repair performed. Only small in-memory counterfactual and saved-parent replay probes.','Saved array integrity exhaustive; generator causality tested across deterministic cases and inspected source, not every possible parameter setting.','Original v2 generator, audit case IDs, and catalog/split/noise dependencies were read only because required to reproduce controls and verify source associations.']
with (OUT/'final_verification_results.json').open('x',encoding='utf-8') as f:json.dump(R,f,indent=2,allow_nan=False)
lines=['# Independent final CNN EEW v3 verification','', '**Date:** 2026-11-13', '', '**Decision: '+R['decision']+'**','', 'All 16,178 HDF5 records were checked directly. All parent samples and numeric arrays were checked for finiteness; every saved raw, normalized and pure-post-P window was recomputed from its parent. Labels/magnitudes were joined to catalog and split sources. No previously reported PASS was used as a substitute for execution.','', '| Check | Status | Evidence |','|---|---|---|']
for name,v in R['checks'].items():lines.append('| '+name+' | '+v['status']+' | '+str(v['evidence']).replace('|','/').replace('\n',' ')+' |')
lines+=['','## Causality evidence','',f"Original v2 regression maxima: noise {maxima['noise']:.9f}; P coda {maxima['coda']:.9f}. Independently injected future-only changes reproduce both reported defects. V3: {len(tests)} earthquake tests across 15 configurations/seeds and {len(nt)} noise tests across four profiles; strict zero tolerance. Non-duration earthquake interventions also require a changed future tail so a no-op cannot pass.",'',f"Saved simulator linkage: {len(replays)} stratified full-parent replays; individual results and source hashes are in the JSON. All observed prefix differences and failures are recorded rather than repaired.",'','## Scientific interpretation','', 'Oracle P alignment is explicit. A real deployment would need a causal trigger and an evaluation without oracle timing. Use waveform arrays only; do not feed labels, masks, IDs or P/S metadata into the CNN. Apply classification_mask_{h}s for classification and magnitude_mask_{h}s for regression at each horizon. The base magnitude_mask alone does not enforce pre-S eligibility. All windows contain 100 pre-P context samples and 100/200/300 post-P samples. Noise references are synthetic random anchors. Physical amplitudes are relative simulator amplitudes, not calibrated instrument displacement. Real-noise/domain validity, magnitude identifiability, false-alert rates and operational latency remain outside this dataset verification.','', '## Remaining check','', 'The requested NPZ verification is UNKNOWN: no v3 NPZ artifact is present. Supply its existing location for an independent array/row/mask comparison, or explicitly scope approval to the verified HDF5 artifacts. No archive was created or repaired by this verifier.','', '## Preservation and reproducibility','', 'All executable verification logic is in final_verification.ipynb. The notebook executes only isolated simulator definitions; original write/generation cells are never executed. Input hashes were checked before/after. Only this new verification directory contains outputs. No CNN training or Optuna was started.']
with (OUT/'final_verification_report.md').open('x',encoding='utf-8') as f:f.write('\n'.join(lines)+'\n')
print(R['decision'])


CONDITIONAL — NPZ artifact verification remains outstanding
